For time-series portfolio returns, I would use a moving block bootstrap rather than ordinary np.random.choice(). The key is that consecutive observations stay together inside each block.

In [1]:
import numpy as np
import pandas as pd


# ============================================================
# 1. Example time-series portfolio returns
# ============================================================

returns = np.array([
    0.012, 0.015, 0.010,      # Block 1
   -0.002, -0.005, -0.008,    # Block 2
    0.003,  0.004,  0.006,    # Block 3
   -0.001,  0.002,  0.005,    # Block 4
    0.008,  0.011,  0.009     # Block 5
])


# ============================================================
# 2. Moving Block Bootstrap
# ============================================================

def moving_block_bootstrap(
    data,
    block_size=3,
    n_bootstrap=10000,
    random_state=42
):
    """
    Moving Block Bootstrap for time-series data.

    Parameters
    ----------
    data : array-like
        Original time-series observations.

    block_size : int
        Number of consecutive observations in each block.

    n_bootstrap : int
        Number of bootstrap samples.

    random_state : int
        Seed for reproducibility.

    Returns
    -------
    bootstrap_samples : ndarray
        Resampled time-series data.
    """

    data = np.asarray(data)

    rng = np.random.default_rng(random_state)

    n = len(data)

    # --------------------------------------------------------
    # Create overlapping consecutive blocks
    #
    # Example:
    # block_size = 3
    #
    # [1,2,3]
    # [2,3,4]
    # [3,4,5]
    # ...
    # --------------------------------------------------------

    blocks = np.array([
        data[i:i + block_size]
        for i in range(n - block_size + 1)
    ])

    bootstrap_samples = []

    # Number of blocks required to approximately
    # reproduce the original sample size
    n_blocks = int(np.ceil(n / block_size))

    for _ in range(n_bootstrap):

        # Randomly select blocks WITH replacement
        selected_indices = rng.integers(
            low=0,
            high=len(blocks),
            size=n_blocks
        )

        selected_blocks = blocks[selected_indices]

        # Flatten blocks
        sample = selected_blocks.flatten()

        # Keep exactly the original sample size
        sample = sample[:n]

        bootstrap_samples.append(sample)

    return np.array(bootstrap_samples)


# ============================================================
# 3. Run bootstrap
# ============================================================

bootstrap_samples = moving_block_bootstrap(
    returns,
    block_size=3,
    n_bootstrap=10000
)

print("Original data:")
print(returns)

print("\nBootstrap sample:")
print(bootstrap_samples[0])

print("\nShape:")
print(bootstrap_samples.shape)


# ============================================================
# 4. Calculate a statistic for every bootstrap sample
# ============================================================

bootstrap_means = np.mean(
    bootstrap_samples,
    axis=1
)


# ============================================================
# 5. Bootstrap confidence interval
# ============================================================

lower = np.percentile(
    bootstrap_means,
    2.5
)

upper = np.percentile(
    bootstrap_means,
    97.5
)

original_mean = np.mean(returns)


print("\n========== BLOCK BOOTSTRAP ==========")

print(
    f"Original mean return : {original_mean:.4%}"
)

print(
    f"95% Bootstrap CI     : "
    f"[{lower:.4%}, {upper:.4%}]"
)


# ============================================================
# 6. Decision
# ============================================================

if lower > 0:

    print("\nDecision:")
    print("The 95% confidence interval is entirely above zero.")
    print(
        "There is evidence that the mean return is positive."
    )

elif upper < 0:

    print("\nDecision:")
    print("The 95% confidence interval is entirely below zero.")
    print(
        "There is evidence that the mean return is negative."
    )

else:

    print("\nDecision:")
    print("The 95% confidence interval contains zero.")
    print(
        "There is insufficient evidence to conclude that "
        "the mean return is different from zero."
    )

Original data:
[ 0.012  0.015  0.01  -0.002 -0.005 -0.008  0.003  0.004  0.006 -0.001
  0.002  0.005  0.008  0.011  0.009]

Bootstrap sample:
[ 0.015  0.01  -0.002  0.002  0.005  0.008  0.006 -0.001  0.002 -0.008
  0.003  0.004 -0.008  0.003  0.004]

Shape:
(10000, 15)

========== BLOCK BOOTSTRAP ==========
Original mean return : 0.4600%
95% Bootstrap CI     : [-0.0600%, 0.7733%]

Decision:
The 95% confidence interval contains zero.
There is insufficient evidence to conclude that the mean return is different from zero.


**For comparing two portfolio strategies**

If your actual objective is:
**Does Strategy A have a different mean return from Strategy B while preserving time dependence?'**
you can bootstrap the difference in means:

In [2]:
def block_bootstrap_difference(
    strategy_a,
    strategy_b,
    block_size=20,
    n_bootstrap=10000,
    random_state=42
):

    strategy_a = np.asarray(strategy_a)
    strategy_b = np.asarray(strategy_b)

    if len(strategy_a) != len(strategy_b):
        raise ValueError(
            "Both strategies must have the same number of observations."
        )

    rng = np.random.default_rng(random_state)

    n = len(strategy_a)

    # --------------------------------------------------------
    # Create overlapping blocks for each strategy
    # --------------------------------------------------------

    blocks_a = np.array([
        strategy_a[i:i + block_size]
        for i in range(n - block_size + 1)
    ])

    blocks_b = np.array([
        strategy_b[i:i + block_size]
        for i in range(n - block_size + 1)
    ])

    n_blocks = int(np.ceil(n / block_size))

    differences = []

    for _ in range(n_bootstrap):

        # Sample blocks independently
        indices_a = rng.integers(
            0,
            len(blocks_a),
            size=n_blocks
        )

        indices_b = rng.integers(
            0,
            len(blocks_b),
            size=n_blocks
        )

        sample_a = blocks_a[indices_a].flatten()[:n]
        sample_b = blocks_b[indices_b].flatten()[:n]

        # Difference in mean returns
        difference = (
            np.mean(sample_a)
            - np.mean(sample_b)
        )

        differences.append(difference)

    return np.array(differences)


# ============================================================
# Example
# ============================================================

strategy_a = np.array([
    0.012, 0.015, 0.010,
   -0.002, -0.005, -0.008,
    0.013, 0.011, 0.009,
   -0.001, 0.004, 0.006,
    0.015, 0.012, 0.010
])

strategy_b = np.array([
    0.005, 0.007, 0.006,
    0.001, -0.001, -0.002,
    0.006, 0.005, 0.004,
    0.001, 0.003, 0.004,
    0.007, 0.006, 0.005
])


# Run block bootstrap
bootstrap_diff = block_bootstrap_difference(
    strategy_a,
    strategy_b,
    block_size=3,
    n_bootstrap=10000
)


# Observed difference
observed_difference = (
    np.mean(strategy_a)
    - np.mean(strategy_b)
)


# 95% CI
lower = np.percentile(
    bootstrap_diff,
    2.5
)

upper = np.percentile(
    bootstrap_diff,
    97.5
)


print("\n========== STRATEGY COMPARISON ==========")

print(
    f"Strategy A mean : {np.mean(strategy_a):.4%}"
)

print(
    f"Strategy B mean : {np.mean(strategy_b):.4%}"
)

print(
    f"Observed difference : {observed_difference:.4%}"
)

print(
    f"95% Block Bootstrap CI : "
    f"[{lower:.4%}, {upper:.4%}]"
)


# ============================================================
# Decision
# ============================================================

if lower > 0:

    print("\nDecision: Reject H0")
    print(
        "Evidence suggests Strategy A has a higher "
        "mean return than Strategy B."
    )

elif upper < 0:

    print("\nDecision: Reject H0")
    print(
        "Evidence suggests Strategy A has a lower "
        "mean return than Strategy B."
    )

else:

    print("\nDecision: Fail to reject H0")
    print(
        "The confidence interval contains zero."
    )

    print(
        "There is insufficient evidence to conclude "
        "that the strategies have different mean returns."
    )


========== STRATEGY COMPARISON ==========
Strategy A mean : 0.6733%
Strategy B mean : 0.3800%
Observed difference : 0.2933%
95% Block Bootstrap CI : [-0.2600%, 0.6933%]

Decision: Fail to reject H0
The confidence interval contains zero.
There is insufficient evidence to conclude that the strategies have different mean returns.
